# Example 8.1 — Schema discovery: what tables are in BindingDB?

A reproducible record of an AI-assisted BindingDB analysis: each prompt, the BindingDB tool calls that answered it with their recorded results, the answer, and its figures. Run all cells to show the recorded results, or rerun every call on your own copy of the database and check each result against the recorded one.

- **Date:** 2026-11-23
- **Model:** claude-opus-5-5
- **Data:** BindingDB release **202611** (`BDB-mySQL_All_202611_dmp.zip`, SHA-256 `daea9260d404919a183ce02d91bf4b9c49b410bea7eaaeba105c8e0280510381`)
- **Database build:** 2026-11-21T01:08:15+00:00, DuckDB 1.5.6
- **Derived data:** standard InChIKeys computed from SMILES by RDKit 2026.03.6; compound_property computed from SMILES (largest fragment) by RDKit 2026.03.6
- **Server:** mcp-bindingdb v0.1.17, commit [`79dae5d22c0f`](https://github.com/sbl-sdsc/mcp-bindingdb/commit/79dae5d22c0f15b8ded9b49adc8b105f0463703a)
- **Software:** python 3.12.7, duckdb 1.5.6, rdkit 2026.03.6, mcp 2.3.0, gemmi 0.7.5
- **Notebook generated:** 2026-11-24T04:56:21Z
- **Contents:** 1 exchange · 0 tool calls

### How to run

Keep the notebook in its folder, which holds everything it needs: `requirements.txt` (mcp-bindingdb at its commit and the versions of the packages it needs) and the recorded results in `data/`. With [uv](https://docs.astral.sh/uv/), from that folder (replace the path with yours):

```bash
cd /path/to/8.1-schema-discovery-notebook
uv run --no-project --with-requirements requirements.txt --with jupyterlab jupyter lab 8.1-schema-discovery-notebook.ipynb
```

then choose Run → Run All Cells. The first start installs the packages, which takes a minute or two; the cells show the recorded results, so no database is needed. Without uv: `pip install -r requirements.txt jupyterlab`, then `jupyter lab 8.1-schema-discovery-notebook.ipynb` in the folder.

### How to rerun the calls

To rerun every BindingDB call on your own copy of release 202611 and check each result against the recorded one, set `RERUN = True` in the setup cell and run all cells, or rerun the notebook without opening Jupyter, writing a copy with the new results:

```bash
cd /path/to/8.1-schema-discovery-notebook
BINDINGDB_RERUN=1 uv run --no-project --with-requirements requirements.txt --with nbconvert jupyter nbconvert --to notebook --execute --output 8.1-schema-discovery-notebook-rerun.ipynb 8.1-schema-discovery-notebook.ipynb
```

A rerun needs the database: the one `BINDINGDB_DUCKDB=/path/to/bindingdb_202611.duckdb` names (put it before `uv run`), else the newest one mcp-bindingdb built on this computer, for example for Claude Desktop. With neither, the first rerun builds one, which takes about half an hour and 2.4 GB. To build exactly release 202611, with a clone of mcp-bindingdb:

```bash
git clone https://github.com/sbl-sdsc/mcp-bindingdb.git
cd mcp-bindingdb
git checkout 79dae5d22c0f15b8ded9b49adc8b105f0463703a
scripts/build_duckdb.sh --release 202611
```

The setup cell checks the database's release and source checksum against the recorded ones.

Run as it is, each call cell shows its recorded result, read from `data/` and checked against the SHA-256 digest recorded in the cell. Rerun, it shows the new result with ✅ when it matches the recorded one (floats to 10 significant digits, ids of viewers and files left out). Calls to PubChem, RCSB or PDBe depend on those services and may differ. The prose answers are the model's and are not recomputed.

In [ ]:
# Shows the results recorded during the session (data/), so the notebook runs without a database. Set
# RERUN = True (or start Jupyter with BINDINGDB_RERUN=1) to rerun every BindingDB call on a local copy of
# the database instead and check it against its recorded result: see 'How to rerun the calls' above.
from mcp_bindingdb.notebook import Replay

RERUN = False
replay = Replay(release='202611', source_zip_sha256='daea9260d404919a183ce02d91bf4b9c49b410bea7eaaeba105c8e0280510381', rerun=RERUN)

Recorded with BindingDB release 202611, mcp-bindingdb v0.1.17 (commit 79dae5d22c0f)
python 3.12.7, duckdb 1.5.6, rdkit 2026.03.6, mcp 2.3.0, gemmi 0.7.5


## Exchange 1

**👤 User**

What tables are in the BindingDB database, and which ones should I start with?

**🧠 Assistant**

The database has **38 tables**: 37 from BindingDB plus a small `_build_info` table that records which release it was built from (here **202611**). Start with the six **derived** tables. They were built from the raw BindingDB tables to make queries easy: values are parsed into numbers with explicit units, compound, target and citation details are copied onto every measurement row, and compound properties are computed once with RDKit.

**Start here (derived tables)**

| Table | Rows | One row per |
|---|---:|---|
| `activity` | 3,246,372 | measurement (Ki, Kd, IC50, EC50, kon, koff) with its compound, target, assay and citation |
| `compound` | 1,477,577 | small molecule: name, InChIKey, SMILES, formula, molecular weight, activity/target counts |
| `compound_property` | 1,466,457 | compound (join on `monomerid`): RDKit heavy atoms, MW, cLogP, TPSA, HBD/HBA, rotatable bonds, rings, Fsp3, QED, Ro5 violations, Bemis-Murcko scaffold, PAINS/Brenk/NIH alerts |
| `compound_name` | 2,552,626 | compound synonym (names, ChEMBL ids, PubChem cid_N, patent labels) |
| `target` | 13,419 | target (protein or complex): UniProt, organism, counts |
| `target_name` | 78,260 | target synonym (display names, gene names, UniProt entry names) |

**Raw tables you may need later.** These mirror BindingDB's own MySQL schema:

| Table | Rows | Use it for |
|---|---:|---|
| `article` | 57,282 | publications and patents: PubMed id or patent number, DOI, title, year |
| `assay` | 233,370 | full assay names and descriptions |
| `entry` | 57,301 | deposition records: title, measurement technique, date |
| `polymer` / `complex` / `complex_component` | 12,162 / 1,259 / 3,250 | protein sequences and PDB ids; multi-subunit targets and their parts |
| `monomer` | 1,477,577 | raw compound records, including the PDB ligand ids |
| `ki_result` / `enzyme_reactant_set` | 3,241,650 / 3,241,671 | the raw measurements (values stored as text) and the compound-to-target links |
| `itc_result_a_b_ab` | 1,822 | isothermal titration calorimetry: ΔG, ΔH, ΔS, stoichiometry |
| `pdb_bdb` | 185,286 | PDB entry to BindingDB id mappings |
| `cobweb_bdb` | 557,304 | the best-affinity target–inhibitor summary the BindingDB website uses |

The remaining tables describe authors, journals and keywords, and experimental conditions (buffers, solvents, solutes, instruments, data-fitting methods, ITC runs).

**The `activity` table, column by column** (27 columns):

- **The measurement:** `affinity_type`, `relation` ('=', '<', '>'), `value` and `unit` (nM), `p_affinity` (−log10 molar), `raw_value` (the text as deposited), `uncertainty`, `ph`, `temp_k`
- **The compound:** `monomerid`, `compound_name`, `inchi_key`
- **The target:** `polymerid` or `complexid`, `target_name`, `uniprot`, `uniprot_raw` (with mutations and residue ranges), `organism`, `taxid`
- **Where it came from:** `assay_name`, `entryid`, `articleid`, `source_id` (PubMed id or patent number), `doi`, `year`
- **Internal keys:** `ki_result_id`, `reactant_set_id`

**A lesson about derived vs raw tables.** In the raw `ki_result` table an affinity is stored as text, such as ">10000", with the unit implied. In `activity` the same value is split into `relation`, a numeric `value` in nM and a `p_affinity`. That split is what makes `median(p_affinity)` or `WHERE value <= 100` possible. Likewise, `compound_property` saves you from computing descriptors yourself: ligand efficiency is `1.37 * p_affinity / heavy_atoms` and LipE is `p_affinity - clogp`, by joining it to `activity` on `monomerid`. Use the raw tables only when you need something the derived ones leave out, such as ITC thermodynamics (Example 8.5) or PDB mappings (Example 8.6).

**How this was answered**

- `describe_tables` with no argument lists every table with its description and approximate row count.
- `describe_tables` with `table='activity'` lists that table's columns and types, plus a few sample rows (likewise for `compound_property` and `_build_info`).
- `describe_tables` only reads the schema, so it isn't recorded as a tool call. Use it before writing any `run_sql` query.

**Try next**

- "Which 10 human targets have the most distinct compounds measured?" (Example 8.2)
- "Describe the `compound_property` table" — then join it to `activity` for ligand efficiency

## Verification

How each rerun call compared with its recorded result.

In [ ]:
replay.summary()

_Recorded: 0 tool calls. Run all cells to compare._

## How to cite

Cite this notebook (for example deposited with a DOI) together with the data and software it ran on:

- Data: BindingDB release 202611, https://www.bindingdb.org/rwd/bind/BDB-mySQL_All_202611_dmp.zip (SHA-256 daea9260d404919a183ce02d91bf4b9c49b410bea7eaaeba105c8e0280510381)
- Software: mcp-bindingdb v0.1.17, commit 79dae5d22c0f15b8ded9b49adc8b105f0463703a, https://github.com/sbl-sdsc/mcp-bindingdb
- Analysis: Example 8.1 — Schema discovery: what tables are in BindingDB?, 2026-11-23, with claude-opus-5-5
- BindingDB: Liu T, Hwang L, Burley SK, Nitsche CI, Southan C, Walters WP, Gilson MK. BindingDB in 2024: a FAIR knowledgebase of protein-small molecule binding data. Nucleic Acids Research 53(D1):D1633-D1644, 2025. https://doi.org/10.1093/nar/gkae1075

As BibTeX, for this server, BindingDB and RDKit (used for structure search, properties and drawings):

```bibtex
@software{rose2026mcp-bindingdb,
  title={MCP BindingDB Server},
  author={Rose, P.W.},
  year={2026},
  url={https://github.com/sbl-sdsc/mcp-bindingdb}
}

@article{liu2025bindingdb,
  title={BindingDB in 2024: a FAIR knowledgebase of protein-small molecule binding data},
  author={Liu, T. and Hwang, L. and Burley, S.K. and Nitsche, C.I. and Southan, C. and Walters, W.P. and Gilson, M.K.},
  journal={Nucleic Acids Research},
  volume={53},
  number={D1},
  pages={D1633--D1644},
  year={2025},
  doi={10.1093/nar/gkae1075}
}

@software{rdkit,
  title={RDKit: Open-source cheminformatics},
  author={Landrum, G. and others},
  version={2026.03.6},
  url={https://www.rdkit.org},
  doi={10.5281/zenodo.591637}
}
```